In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://marina-distance-drapery.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://marina-distance-drapery.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大（MUST），是臺灣一所私立科技大學，位於新竹縣新豐鄉。學校秉持「堅毅、求新、創造」的校訓，致力於培育具備實務經驗與人文素養的專業人才，並以成為「國際魅力產業科技大學」為願景。

**發展沿革**
明新科技大學的歷史可追溯至1966年創立的「明新工業專科學校」。1997年改制為「明新技術學院」並附設專科部，最終於2002年升格為「明新科技大學」。2018年，學校更名為「明新學校財團法人明新科技大學」。

**辦學特色**
明新科大毗鄰新竹科學園區與新竹工業區，地理位置優越，使其能與產業界建立密切的產學合作關係。 學校積極響應國家重點產業發展，特別在半導體領域表現卓越。根據1111人力銀行統計，明新科大在半導體產業界最受青睞的畢業生中名列前茅，是唯一入榜的私立科技大學，與頂尖大學齊名。 學校斥資兩億元打造「半導體基地」，設有半導體封裝測試類產線，並成立技職體系中的第一座「半導體學院」，甚至獲教育部核准招收「半導體科技博士學位學程」博士生。

明新科大致力於培養學生成為跨領域的專業人才，發展MUST四大育才特色：多元學習（Multidisciplinary Learning）、全球視野（Universal Perspective）、永續經營（Sustainable Operations）與技術創新（Technological Innovation）。 學校推動國際化，擁有全國第一的國際學生人數，學生來自超過16個國家，打造多元的校園環境。 為提升學生的外語能力，學校結合AI科技開發「APEC國際會議模擬混合實境（MR）教室」，並與多所國際大學簽訂雙聯學位及學術交流合作，拓展學生的國際視野與移動力。

**學院與學系**
明新科技大學設有六大學院，包括：半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院。 涵蓋多個學系、碩士班及學位學程，提供多元的學習選擇。

**校訓與願景**
學校以《大學》中「在明明德，在新民，在止於至善」的精義為名，期許學子涵養高尚品德，擁有專業學問與優良技術，達到全人發展的境界。 明新科技大學的願景是成為「國際魅力產業科技大學」，目標是培育具備「跨域整合、務實創新、全人學習」能力的專業人才。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長為**呂明峯教授**。 他於2025年2月1日正式上任，並在2025年1月16日舉行了第11任校長布達暨交接典禮。 呂明峯校長在明新科大服務長達34年，擁有豐富的業界經驗和學術背景，曾任電子工程系主任、研發長、工學院院長、半導體學院院長以及產學長等職務。 他在任內開辦全國首見的「2+2N封裝測試產業精英專班」，並推動成立了台灣首座半導體封裝測試類產線以及半導體學院，甚至獲教育部核准招收「半導體科技博士學位學程」博士生，為明新科大在半導體教育領域樹立了顯著的招牌。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ua6692ca970e3a5c8679351bf1eb1a473","events":[{"type":"message","message":{"type":"text","id":"615036592738533634","quoteToken":"JJsQX0BeMYcJjWsJM6IV5pCvHNE3YB2kM_El3JESUuTZjhWeyM9eqovY-COzR2BGwO05clJHSq_b3iHuryx_9ogd6PDv4niS2MT_8t7dBOW6OxOKR27V92F8-jQ1SAnBZlDsc9CVKTJ45MsM3yiBLA","markAsReadToken":"CcciDlreeWFMI8YpGWctlw2trLV0phkBFNPig3m-g9-sxJavELxRXIVULoTgCd4rFT5qL_KseojjacN7xwKkbZ5BC237IkiWLPdJjh35ANcIvy_7vn5_U21M8ladcDUbxhpq8SvbDr2cfWryvonNLkJ5oSuozVLmqXmea2_EHBxx6_LgPJAvIQTArbdg348ACbh2R8PhniXUGNc4ryAWyQ","text":"AI 校長是誰"},"webhookEventId":"01KS6WR50V6TV2V1VJYSTXFKHW","deliveryContext":{"isRedelivery":false},"timestamp":1779421680413,"source":{"type":"user","userId":"Ueea9bd8ab131bc5c9ade769c026be174"},"replyToken":"eacd920a9dd446c49b2dd57051c9c90d","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:48:02] "POST / HTTP/1.1" 200 -
